# TruthLens AI: Model Training (Google Colab)

**How to run**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Check the settings in the **CONFIG** cell below
3. **Runtime → Run all**

No Kaggle account, API key or login is needed. All datasets and models download automatically:

| Model | Data | License |
|---|---|---|
| English: DeBERTa-v3-large (+ BERT baseline) | WELFake, 72k news articles (Zenodo) | CC BY 4.0 |
| Telugu: IndicBERT v2 | WELFake translated to Telugu with NLLB-200 (built in this notebook) | CC BY 4.0 / CC BY-NC 4.0 |
| Image: CLIP ViT-L/14 | CIFAKE, 100k images (Hugging Face) | MIT |

**Tip:** run once with `QUICK_TEST = True` (about 25 minutes) to check that everything works. Then set it to `False` for the real training.

## ⚙️ CONFIG

In [ ]:
QUICK_TEST = True        # True = small, fast trial run. False = real training for the report and paper.

TRAIN_IMAGE    = True    # CLIP ViT-L/14 image detector               (~20-40 min)
TRAIN_ENGLISH  = True    # DeBERTa-v3-large, English, main model      (~1.5-2.5 h)
TRAIN_BASELINE = True    # BERT-base, English, for comparison table   (~30-60 min)
TRAIN_TELUGU   = True    # translate news to Telugu + IndicBERT v2    (~30-40 min + ~20 min)

# Models (all free on Hugging Face, no login)
EN_MODEL          = "microsoft/deberta-v3-large"
BASELINE_MODEL    = "bert-base-uncased"
TE_MODEL          = "ai4bharat/IndicBERTv2-MLM-only"
CLIP_MODEL        = "openai/clip-vit-large-patch14"
TRANSLATION_MODEL = "facebook/nllb-200-distilled-1.3B"

# Real-training sizes (limits keep each run inside one free Colab session)
EN_MAX_TRAIN_ROWS = 20000   # WELFake has ~64k clean rows; 20k gives strong results in ~2 h. 0 = use all.
TE_ROWS           = 10000   # English articles translated to Telugu (half fake, half real)
IMG_MAX_PER_CLASS = 20000   # CIFAKE has 50k per class. 0 = use all.

## 1. Setup: Google Drive, GPU check, folders

In [ ]:
import os, json, glob, shutil

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    # Kaggle: everything in /kaggle/working is kept as the notebook's Output after "Save & Run All".
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one GPU (same tested setup as Colab T4)
    BASE, WORK = "/kaggle/working/truthlens", "/tmp/truthlens"
    print("✅ Running on Kaggle: results are saved in the notebook's Output tab.")
else:
    WORK = "/content"
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = "/content/drive/MyDrive/truthlens"
        print("✅ Google Drive mounted: trained models are kept even if Colab disconnects.")
    except Exception as e:
        BASE = "/content/truthlens"
        print("⚠️ Google Drive not available:", e)
        print("   Models are saved on the Colab machine. Download the zip at the end before closing!")

DATA  = f"{WORK}/data"   # downloaded datasets live on the fast local disk (re-downloaded each session)
OUT   = f"{WORK}/outputs_quick" if QUICK_TEST else f"{BASE}/outputs"
CKPT  = f"{WORK}/checkpoints"    # temporary training checkpoints (local disk)
CACHE = f"{BASE}/cache"           # slow-to-build data (Telugu translation) is kept on Drive
for d in (BASE, DATA, OUT, CKPT, CACHE):
    os.makedirs(d, exist_ok=True)
os.chdir(BASE)

import torch
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ NO GPU! Colab: Runtime → Change runtime type → T4 GPU. "
          "Kaggle: Settings → Accelerator → GPU T4 x2 (and Internet on). Then run again.")
print("Mode:", "QUICK TEST" if QUICK_TEST else "FULL TRAINING", "| outputs →", OUT)


def run(cmd):
    """Run a shell command, show its output live, stop the notebook if it fails."""
    import sys
    if cmd.startswith("python "):   # always use this notebook's own Python
        cmd = f'"{sys.executable}" ' + cmd[len("python "):]
    elif cmd.startswith("pip "):
        cmd = f'"{sys.executable}" -m pip ' + cmd[len("pip "):]
    print("$", cmd)
    ip = get_ipython()
    ip.system(cmd)
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Command failed: {cmd}")


def extract(zip_file, dest):
    """Unzip with Python (works everywhere), skipping macOS junk files."""
    import zipfile
    print("unzipping", zip_file, "→", dest)
    with zipfile.ZipFile(zip_file) as z:
        z.extractall(dest, members=[m for m in z.namelist() if not m.startswith("__MACOSX")])


def done(name):
    """True if this model was already trained (lets you re-run after a disconnect)."""
    if os.path.exists(f"{OUT}/{name}/metrics.json"):
        print(f"⏭️  {name} already trained. Skipping (delete {OUT}/{name} to retrain).")
        return True
    return False

## 2. Training scripts (written automatically) + packages

In [ ]:
%%writefile requirements.txt
# Training dependencies (Google Colab / Kaggle already have torch installed)
transformers>=4.48,<5.0
huggingface-hub>=0.34,<1.0
datasets>=2.19
accelerate>=0.30
sentencepiece>=0.2.0
protobuf>=4.25
scikit-learn>=1.4
pandas>=2.1
matplotlib>=3.8
pillow>=10.0
tqdm>=4.66

In [ ]:
%%writefile prepare_dataset.py
"""Convert a fake-news dataset into the standard format used by train_text_classifier.py.

Output CSV columns:  text,label     (label: 0 = Real, 1 = Fake)

Examples
--------
ISOT (two files, Fake.csv and True.csv):
    python prepare_dataset.py isot --fake Fake.csv --real True.csv --out data/en.csv

Any single CSV (WELFake, Telugu datasets, your own data). Say which value means FAKE:
    python prepare_dataset.py generic --input WELFake_Dataset.csv --text-cols title text \
        --label-col label --fake-value 1 --remove-patterns welfake --out data/en.csv
    (WELFake: the actual file has 1 = fake, 0 = real - the opposite of its documentation.
     Always check a few rows yourself: Reuters / NYT articles must end up as label 0 = Real.)

Folders of .txt files (e.g. the Zenodo Telugu dataset, CC BY 4.0, DOI 10.5281/zenodo.11408513):
    python prepare_dataset.py folders --fake-dir data/te/Telugu_fake_news --real-dir data/te/Telugu_real_news --out data/te.csv

Split into train / test (stratified, 80/20):
    python prepare_dataset.py split --input data/en.csv --test-size 0.2
"""
import argparse
import re
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


# Source giveaways that let a model "cheat" (it learns the publisher, not whether the news is fake).
PATTERN_PRESETS = {
    "welfake": [
        r"^[^\n]{0,120}?\(Reuters\)\s*[-\u2013\u2014]?\s*",   # "WASHINGTON (Reuters) - " dateline
        r"\(?\bReuters\b\)?",
        r"\s[-\u2013\u2014]\s(The New York Times|Breitbart|The Onion)\b",
        r"Featured (image|photo)[^.\n]{0,120}",
        r"\b(Getty Images|21st Century Wire|21WIRE)\b",
        r"\bBreitbart( News)?\b",
        r"Follow (him|her|them|us|me)?\s*on Twitter[^.\n]{0,80}",
    ],
}


def remove_patterns(series: pd.Series, patterns: list[str]) -> pd.Series:
    for pat in patterns:
        for regex in PATTERN_PRESETS.get(pat, [pat]):
            series = series.str.replace(regex, " ", regex=True, flags=re.IGNORECASE | re.MULTILINE)
    return series


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["text"].str.len() >= 20]
    df = df.drop_duplicates(subset="text")
    return df.sample(frac=1, random_state=42).reset_index(drop=True)


def join_cols(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    return df[cols].fillna("").astype(str).agg(". ".join, axis=1)


def report(df: pd.DataFrame, out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    df[["text", "label"]].to_csv(out, index=False, encoding="utf-8")
    counts = df["label"].value_counts().to_dict()
    print(f"Saved {len(df)} rows to {out}  (Real={counts.get(0, 0)}, Fake={counts.get(1, 0)})")


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    isot = sub.add_parser("isot")
    isot.add_argument("--fake", required=True)
    isot.add_argument("--real", required=True)
    isot.add_argument("--out", required=True)

    gen = sub.add_parser("generic")
    gen.add_argument("--input", required=True)
    gen.add_argument("--text-cols", nargs="+", required=True)
    gen.add_argument("--label-col", required=True)
    gen.add_argument("--fake-value", required=True, help="value in label column that means FAKE")
    gen.add_argument("--out", required=True)
    gen.add_argument("--remove-patterns", nargs="*", default=[],
                     help="regexes (or preset name 'welfake') to delete from every text")
    gen.add_argument("--max-rows", type=int, default=0, help="keep a balanced random sample (0 = all)")

    folders = sub.add_parser("folders")
    folders.add_argument("--fake-dir", required=True)
    folders.add_argument("--real-dir", required=True)
    folders.add_argument("--out", required=True)
    folders.add_argument("--remove-words", nargs="*", default=[],
                         help="words to delete from every text (stops the model cheating on giveaway words)")

    split = sub.add_parser("split")
    split.add_argument("--input", required=True)
    split.add_argument("--test-size", type=float, default=0.2)

    args = parser.parse_args()

    if args.cmd == "isot":
        fake, real = pd.read_csv(args.fake), pd.read_csv(args.real)
        fake["label"], real["label"] = 1, 0
        df = pd.concat([fake, real], ignore_index=True)
        df["text"] = join_cols(df, ["title", "text"])
        report(clean(df), Path(args.out))

    elif args.cmd == "generic":
        df = pd.read_csv(args.input)
        df["text"] = join_cols(df, args.text_cols)
        df["label"] = (df[args.label_col].astype(str).str.strip() == str(args.fake_value)).astype(int)
        if args.remove_patterns:
            df["text"] = remove_patterns(df["text"], args.remove_patterns)
            print(f"Removed source giveaways: {', '.join(args.remove_patterns)}")
        df = clean(df)
        if args.max_rows and len(df) > args.max_rows:
            per = args.max_rows // 2
            df = pd.concat([g.sample(n=min(per, len(g)), random_state=42) for _, g in df.groupby("label")])
            df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        report(df, Path(args.out))

    elif args.cmd == "folders":
        rows = []
        for label, folder in ((1, args.fake_dir), (0, args.real_dir)):
            files = sorted(Path(folder).rglob("*.txt"))
            if not files:
                raise SystemExit(f"No .txt files found in {folder}")
            for f in files:
                text = f.read_text(encoding="utf-8", errors="replace").replace("|", " ")
                rows.append({"text": text, "label": label})
        df = pd.DataFrame(rows)
        if args.remove_words:
            pattern = "|".join(re.escape(w) for w in args.remove_words)
            df["text"] = df["text"].str.replace(pattern, " ", regex=True)
            print(f"Removed giveaway words: {', '.join(args.remove_words)}")
        report(clean(df), Path(args.out))

    elif args.cmd == "split":
        path = Path(args.input)
        df = pd.read_csv(path)
        train, test = train_test_split(df, test_size=args.test_size, stratify=df["label"], random_state=42)
        train.to_csv(path.with_name(path.stem + "_train.csv"), index=False)
        test.to_csv(path.with_name(path.stem + "_test.csv"), index=False)
        print(f"train: {len(train)} rows, test: {len(test)} rows")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile translate_dataset.py
"""Build a Telugu fake-news dataset by machine-translating a balanced English sample.

Why: the public Telugu fake-news dataset compares *translated fact-check reports* (fake) with *native
health articles* (real), so a model learns the writing style, not fake vs real. Translating fake AND real
English news with the same model gives both classes the same style, so the model must learn the content.

    python translate_dataset.py --input data/en.csv --out data/te.csv --rows 10000

Model: facebook/nllb-200-distilled-1.3B (free, no login, CC BY-NC 4.0: fine for academic use).
Each article is shortened to its headline + first sentences (--max-words) and translated sentence by
sentence. Progress is saved every few batches, so after a disconnect just run the same command again.
"""
import argparse
import re
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SENTENCE_END = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'“])")


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True, help="English CSV with text,label (from prepare_dataset.py)")
    p.add_argument("--out", required=True)
    p.add_argument("--rows", type=int, default=10000, help="balanced number of articles to translate")
    p.add_argument("--max-words", type=int, default=120, help="words kept per article before translating")
    p.add_argument("--model", default="facebook/nllb-200-distilled-1.3B")
    p.add_argument("--batch-size", type=int, default=48)
    p.add_argument("--save-every", type=int, default=20, help="save progress every N batches")
    return p.parse_args()


def shorten(text: str, max_words: int) -> list[str]:
    """Headline + leading sentences, at most max_words words, as a list of sentences."""
    sentences, words = [], 0
    for s in SENTENCE_END.split(str(text).strip()):
        s = s.strip()
        if not s:
            continue
        n = len(s.split())
        if n > 60:  # very long "sentence" (often no punctuation) - cut it
            s, n = " ".join(s.split()[:60]), 60
        if sentences and words + n > max_words:
            break
        sentences.append(s)
        words += n
    return sentences


def main():
    args = parse_args()
    out = Path(args.out)
    partial = out.with_suffix(".partial.csv")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    df = pd.read_csv(args.input)[["text", "label"]].dropna()
    per_class = args.rows // 2
    df = pd.concat([g.sample(n=min(per_class, len(g)), random_state=42) for _, g in df.groupby("label")])
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    df["doc_id"] = range(len(df))

    # one row per sentence
    segments = [(doc, i, s) for doc, text in zip(df.doc_id, df.text) for i, s in enumerate(shorten(text, args.max_words))]
    seg = pd.DataFrame(segments, columns=["doc_id", "sent_id", "en"])

    done = pd.read_csv(partial) if partial.exists() else pd.DataFrame(columns=["doc_id", "sent_id", "te"])
    done_keys = set(zip(done.doc_id, done.sent_id))
    todo = seg[[k not in done_keys for k in zip(seg.doc_id, seg.sent_id)]]
    print(f"{len(df)} articles → {len(seg)} sentences ({len(todo)} left to translate) on {device}")

    if len(todo):
        tok = AutoTokenizer.from_pretrained(args.model, src_lang="eng_Latn")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            args.model, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32
        ).to(device).eval()
        tel = tok.convert_tokens_to_ids("tel_Telu")

        todo = todo.iloc[todo.en.str.len().argsort()]  # similar lengths together = faster batches
        results, batches = [], 0
        for start in tqdm(range(0, len(todo), args.batch_size), desc="Translating"):
            batch = todo.iloc[start : start + args.batch_size]
            enc = tok(list(batch.en), return_tensors="pt", padding=True, truncation=True, max_length=200).to(device)
            with torch.inference_mode():
                gen = model.generate(**enc, forced_bos_token_id=tel, max_new_tokens=256, num_beams=1)
            te = tok.batch_decode(gen, skip_special_tokens=True)
            results.append(pd.DataFrame({"doc_id": batch.doc_id.values, "sent_id": batch.sent_id.values, "te": te}))
            batches += 1
            if batches % args.save_every == 0:
                done = pd.concat([done, *results], ignore_index=True)
                results = []
                done.to_csv(partial, index=False)
        done = pd.concat([done, *results], ignore_index=True)
        done.to_csv(partial, index=False)

    done = done.sort_values(["doc_id", "sent_id"])
    telugu = done.groupby("doc_id")["te"].apply(lambda s: " ".join(map(str, s)))
    result = df.set_index("doc_id").join(telugu.rename("te"), how="inner")
    result = result[result.te.str.len() > 20]
    out.parent.mkdir(parents=True, exist_ok=True)
    result[["te", "label", "text"]].rename(columns={"te": "text", "text": "english_source"}).to_csv(
        out, index=False, encoding="utf-8"
    )
    counts = result.label.value_counts().to_dict()
    print(f"Saved {len(result)} Telugu articles to {out}  (Real={counts.get(0, 0)}, Fake={counts.get(1, 0)})")
    print("Example:", result.te.iloc[0][:200])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bias_check.py
"""Dataset bias check: how well does a simple word-count model (TF-IDF + logistic regression) do?

If a model that does not understand language at all scores ~100%, the dataset has a shortcut
(publisher names, writing style, topic) and a high transformer score means little. Report this
table in the paper's dataset section.

    python bias_check.py --data data/en.csv data/te.csv --out outputs/bias_check.md
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def check(path: str, max_rows: int) -> str:
    df = pd.read_csv(path)[["text", "label"]].dropna()
    if len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=0)
    if len(df) < 20 or df.label.nunique() < 2:
        return f"### {Path(path).name}\n\n- Skipped: only {len(df)} rows (need at least 20 with both labels).\n"
    tr, te = train_test_split(df, test_size=0.2, stratify=df.label, random_state=42)
    vec = TfidfVectorizer(token_pattern=r"\S+", min_df=3 if len(tr) >= 500 else 1, max_features=50000)
    clf = LogisticRegression(max_iter=2000).fit(vec.fit_transform(tr.text), tr.label)
    pred = clf.predict(vec.transform(te.text))
    weights, names = clf.coef_[0], np.array(vec.get_feature_names_out())
    fake = ", ".join(names[np.argsort(-weights)[:12]])
    real = ", ".join(names[np.argsort(weights)[:12]])
    return (f"### {Path(path).name}\n\n"
            f"- Rows used: {len(df)}\n"
            f"- TF-IDF + Logistic Regression: accuracy **{accuracy_score(te.label, pred):.4f}**, "
            f"F1 **{f1_score(te.label, pred):.4f}**\n"
            f"- Strongest *fake* words: {fake}\n"
            f"- Strongest *real* words: {real}\n")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", nargs="+", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--max-rows", type=int, default=30000)
    args = p.parse_args()
    report = "## Dataset bias check (simple word model)\n\n" + "\n".join(check(d, args.max_rows) for d in args.data)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(report, encoding="utf-8")
    print(report)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_text_classifier.py
"""Fine-tune a transformer for Fake / Real news classification.

Main English model (proposed):
    python train_text_classifier.py --model microsoft/deberta-v3-large \
        --train data/en_train.csv --test data/en_test.csv --out outputs/text_en

English baseline (for the comparison table in the report / paper):
    python train_text_classifier.py --model bert-base-uncased \
        --train data/en_train.csv --test data/en_test.csv --out outputs/bert_baseline --lr 2e-5 --batch-size 16

Telugu model:
    python train_text_classifier.py --model ai4bharat/IndicBERTv2-MLM-only \
        --train data/te_train.csv --test data/te_test.csv --out outputs/text_te --lr 3e-5 --batch-size 16

Input CSVs: columns `text,label` with label 0 = Real, 1 = Fake (see prepare_dataset.py).
Outputs in --out: the model (copy this folder to backend/models/...), metrics.json,
classification_report.txt, confusion_matrix.png, training_curve.png.
"""
import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

LABELS = {0: "Real", 1: "Fake"}


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="microsoft/deberta-v3-large")
    p.add_argument("--train", required=True)
    p.add_argument("--test", help="test CSV; if omitted, 15%% of --train is held out")
    p.add_argument("--out", required=True)
    p.add_argument("--max-length", type=int, default=256)
    p.add_argument("--epochs", type=float, default=3)
    p.add_argument("--batch-size", type=int, default=8)
    p.add_argument("--grad-accum", type=int, default=4)
    p.add_argument("--lr", type=float, default=1e-5)
    p.add_argument("--max-train-rows", type=int, default=0, help="limit rows for a quick run (0 = all)")
    p.add_argument("--max-test-rows", type=int, default=0, help="limit test rows for a quick run (0 = all)")
    p.add_argument("--no-fp16", action="store_true", help="disable mixed precision (use if loss becomes NaN)")
    p.add_argument("--checkpoint-dir", default="", help="where temporary checkpoints go (default: <out>/checkpoints)")
    p.add_argument("--seed", type=int, default=42)
    return p.parse_args()


def load(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)[["text", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    df["text"] = df["text"].astype(str)
    return df


def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}


def main():
    args = parse_args()
    set_seed(args.seed)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    train_df = load(args.train)
    if args.max_train_rows:
        train_df = train_df.sample(n=min(args.max_train_rows, len(train_df)), random_state=args.seed)
    if args.test:
        test_df = load(args.test)
    else:
        train_df, test_df = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=args.seed)
    if args.max_test_rows and len(test_df) > args.max_test_rows:
        test_df, _ = train_test_split(test_df, train_size=args.max_test_rows, stratify=test_df["label"],
                                      random_state=args.seed)
    train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=args.seed)
    print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model, num_labels=2, id2label=LABELS, label2id={v: k for k, v in LABELS.items()}
    )

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=args.max_length)

    def to_ds(df):
        return Dataset.from_pandas(df.reset_index(drop=True)).map(tokenize, batched=True, remove_columns=["text"])

    train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)

    use_fp16 = torch.cuda.is_available() and not args.no_fp16
    training_args = TrainingArguments(
        output_dir=args.checkpoint_dir or str(out / "checkpoints"),
        save_only_model=True,
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2,
        gradient_accumulation_steps=args.grad_accum,
        num_train_epochs=args.epochs,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=use_fp16,
        logging_steps=50,
        report_to="none",
        seed=args.seed,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    # ---------- final evaluation on the held-out test set ----------
    pred = trainer.predict(test_ds)
    probs = torch.softmax(torch.tensor(pred.predictions, dtype=torch.float32), dim=-1).numpy()
    y_true, y_pred = pred.label_ids, probs.argmax(-1)
    metrics = compute_metrics((pred.predictions, y_true))
    metrics["roc_auc"] = float(roc_auc_score(y_true, probs[:, 1]))
    metrics = {k: round(float(v), 4) for k, v in metrics.items()}
    metrics.update({"model": args.model, "test_rows": int(len(y_true)), "max_length": args.max_length})
    print(json.dumps(metrics, indent=2))

    # ---------- save model + report artefacts ----------
    trainer.save_model(str(out))
    tokenizer.save_pretrained(str(out))
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out / "classification_report.txt").write_text(
        classification_report(y_true, y_pred, target_names=["Real", "Fake"], digits=4)
    )

    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["Real", "Fake"]).plot(cmap="Blues")
    plt.title(f"Confusion matrix - {args.model.split('/')[-1]}")
    plt.savefig(out / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()

    history = [h for h in trainer.state.log_history if "loss" in h and "epoch" in h]
    evals = [h for h in trainer.state.log_history if "eval_loss" in h]
    if history:
        plt.plot([h["epoch"] for h in history], [h["loss"] for h in history], label="train loss")
        plt.plot([h["epoch"] for h in evals], [h["eval_loss"] for h in evals], "o-", label="validation loss")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.legend()
        plt.savefig(out / "training_curve.png", dpi=200, bbox_inches="tight")
        plt.close()

    import shutil

    shutil.rmtree(args.checkpoint_dir or (out / "checkpoints"), ignore_errors=True)
    print(f"\nDone. Copy the folder '{out}' to backend/models/ (e.g. backend/models/text_en).")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_image_detector.py
"""Train the Real vs AI-generated image detector: frozen CLIP ViT-L/14 + linear head.

Dataset folder layout - either with train/test folders (Kaggle CIFAKE):
    data/cifake/train/REAL/*.jpg     data/cifake/train/FAKE/*.jpg
    data/cifake/test/REAL/*.jpg      data/cifake/test/FAKE/*.jpg
or just one folder per class (Hugging Face CIFAKE) - then 20% is held out as the test set automatically:
    data/cifake/real/*.jpg           data/cifake/fake/*.jpg
Folder names are matched case-insensitively (REAL / real).

    python train_image_detector.py --data data/cifake --out outputs/image_detector

Outputs in --out: config.json + head.pt (copy the folder to backend/models/image_detector),
metrics.json, classification_report.txt, confusion_matrix.png.
"""
import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch import nn
from tqdm import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection

LABELS = ["Real", "AI-Generated"]  # index 0 = real, 1 = fake
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True, help="folder containing train/ and test/, or real/ and fake/")
    p.add_argument("--out", required=True)
    p.add_argument("--clip", default="openai/clip-vit-large-patch14")
    p.add_argument("--real-dir", default="REAL")
    p.add_argument("--fake-dir", default="FAKE")
    p.add_argument("--max-per-class", type=int, default=0, help="limit images per class (0 = all)")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--test-fraction", type=float, default=0.2, help="test share when there is no test/ folder")
    return p.parse_args()


def find_dir(parent: Path, name: str) -> Path | None:
    for child in parent.iterdir() if parent.is_dir() else []:
        if child.is_dir() and child.name.lower() == name.lower():
            return child
    return None


def class_files(split_dir: Path, real: str, fake: str, limit: int) -> dict[int, list[Path]]:
    result = {}
    for label, sub in ((0, real), (1, fake)):
        folder = find_dir(split_dir, sub)
        files = sorted(f for f in folder.rglob("*") if f.suffix.lower() in EXTS) if folder else []
        if not files:
            raise SystemExit(f"No images found in {split_dir / sub}")
        if limit:
            rng = np.random.default_rng(42)
            files = sorted(rng.choice(files, size=min(limit, len(files)), replace=False))
        result[label] = list(files)
    return result


def load_splits(data: Path, real: str, fake: str, limit: int, test_fraction: float):
    train_dir, test_dir = find_dir(data, "train"), find_dir(data, "test")
    if train_dir and test_dir:
        to_items = lambda d: [(f, l) for l, fs in d.items() for f in fs]
        test_limit = max(1, int(limit * test_fraction / (1 - test_fraction))) if limit else 0
        return (to_items(class_files(train_dir, real, fake, limit)),
                to_items(class_files(test_dir, real, fake, test_limit)))
    # no train/test folders: stratified random split
    total = int(limit / (1 - test_fraction)) if limit else 0
    train, test = [], []
    rng = np.random.default_rng(42)
    for label, files in class_files(data, real, fake, total).items():
        files = list(rng.permutation(files))
        n_test = max(1, int(len(files) * test_fraction))
        test += [(f, label) for f in files[:n_test]]
        train += [(f, label) for f in files[n_test:]]
    print(f"No train/test folders - held out {test_fraction:.0%} as the test set.")
    return train, test


@torch.no_grad()
def embed(items, clip, processor, device, batch_size):
    feats, labels = [], []
    for i in tqdm(range(0, len(items), batch_size), desc="CLIP features"):
        batch = items[i : i + batch_size]
        images = [Image.open(f).convert("RGB") for f, _ in batch]
        pixels = processor(images=images, return_tensors="pt")["pixel_values"].to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            emb = clip(pixel_values=pixels).image_embeds.float()
        feats.append(nn.functional.normalize(emb, dim=-1).cpu())
        labels += [l for _, l in batch]
    return torch.cat(feats), torch.tensor(labels)


def main():
    args = parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(42)

    processor = CLIPImageProcessor.from_pretrained(args.clip)
    clip = CLIPVisionModelWithProjection.from_pretrained(args.clip).to(device).eval()

    data = Path(args.data)
    train_items, test_items = load_splits(data, args.real_dir, args.fake_dir, args.max_per_class, args.test_fraction)
    print(f"train images: {len(train_items)}  test images: {len(test_items)}")

    x_train, y_train = embed(train_items, clip, processor, device, args.batch_size)
    x_test, y_test = embed(test_items, clip, processor, device, args.batch_size)

    # hold out 10% of train for validation
    perm = torch.randperm(len(x_train))
    n_val = max(1, len(perm) // 10)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]

    head = nn.Linear(x_train.shape[1], len(LABELS)).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=args.lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    xt, yt = x_train[tr_idx].to(device), y_train[tr_idx].to(device)
    xv, yv = x_train[val_idx].to(device), y_train[val_idx].to(device)

    best_acc, best_state = -1.0, None
    for epoch in range(args.epochs):
        head.train()
        order = torch.randperm(len(xt), device=device)
        for i in range(0, len(order), 256):
            idx = order[i : i + 256]
            opt.zero_grad()
            loss = loss_fn(head(xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val_acc = (head(xv).argmax(-1) == yv).float().mean().item()
        if val_acc > best_acc:
            best_acc, best_state = val_acc, {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        print(f"epoch {epoch + 1:02d}  loss {loss.item():.4f}  val_acc {val_acc:.4f}")

    head.load_state_dict(best_state)
    head.eval()
    with torch.no_grad():
        probs = torch.softmax(head(x_test.to(device)), -1).cpu().numpy()
    y_true, y_pred = y_test.numpy(), probs.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    metrics = {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(p), 4),
        "recall": round(float(r), 4),
        "f1": round(float(f1), 4),
        "roc_auc": round(float(roc_auc_score(y_true, probs[:, 1])), 4),
        "model": args.clip,
        "test_images": int(len(y_true)),
    }
    print(json.dumps(metrics, indent=2))

    torch.save(best_state, out / "head.pt")
    (out / "config.json").write_text(json.dumps({"clip_model": args.clip, "labels": LABELS}, indent=2))
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out / "classification_report.txt").write_text(
        classification_report(y_true, y_pred, target_names=LABELS, digits=4)
    )
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=LABELS).plot(cmap="Blues")
    plt.title("Confusion matrix - CLIP ViT-L/14 detector")
    plt.savefig(out / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()
    print(f"\nDone. Copy the folder '{out}' to backend/models/image_detector")


if __name__ == "__main__":
    main()

In [ ]:
run("pip install -q -r requirements.txt")

## 3. English data: WELFake
**Label note:** WELFake's documentation says 0 = fake, but in the actual file **1 = fake** (Reuters and
NYT articles are label 0). We use `--fake-value 1`. Publisher giveaways like "(Reuters)", "- The New York Times"
and "Featured image via…" are removed so the model can't cheat by recognising the publisher.

In [ ]:
if (TRAIN_ENGLISH or TRAIN_BASELINE or TRAIN_TELUGU) and not os.path.exists(f"{DATA}/en_test.csv"):
    run(f'wget -q --show-progress -O {DATA}/WELFake.csv '
        f'"https://zenodo.org/records/4561253/files/WELFake_Dataset.csv?download=1"')
    run(f"python prepare_dataset.py generic --input {DATA}/WELFake.csv --text-cols title text "
        f"--label-col label --fake-value 1 --remove-patterns welfake --out {DATA}/en.csv")
    run(f"python prepare_dataset.py split --input {DATA}/en.csv")
print("✅ English data ready")

## 4. Telugu data: translate English news with NLLB-200
The public Telugu fake-news dataset compares translated fact-checks (fake) with native health articles
(real), so even a word-count model scores 99.8% on it. Translating fake **and** real English news with the
same model gives both classes the same style, so the model has to learn the content. The result is cached
on Drive, so this runs only once (~30–40 min on a T4).

In [ ]:
if TRAIN_TELUGU and not os.path.exists(f"{DATA}/te_test.csv"):
    te_csv = f"{CACHE}/te_translated{'_quick' if QUICK_TEST else ''}.csv"
    if not os.path.exists(te_csv):
        rows, words, model = (200, 60, "facebook/nllb-200-distilled-600M") if QUICK_TEST else \
                             (TE_ROWS, 120, TRANSLATION_MODEL)
        run(f"python translate_dataset.py --input {DATA}/en_train.csv --out {te_csv} "
            f"--rows {rows} --max-words {words} --model {model}")
    shutil.copy(te_csv, f"{DATA}/te.csv")
    run(f"python prepare_dataset.py split --input {DATA}/te.csv")
print("✅ Telugu data ready")

## 5. Image data: CIFAKE

In [ ]:
if TRAIN_IMAGE and not os.path.exists(f"{DATA}/cifake/real"):
    run(f'wget -q --show-progress -O {DATA}/cifake.zip '
        f'"https://huggingface.co/datasets/yanbax/CIFAKE_autotrain_compatible/resolve/main/train.zip"')
    extract(f"{DATA}/cifake.zip", f"{DATA}/cifake")
print("✅ Image data ready")

## 6. Dataset bias check (for the paper)
A word-count model. If it scores ~100%, the dataset has a shortcut.

In [ ]:
sets = [f"{DATA}/{n}.csv" for n in ("en", "te") if os.path.exists(f"{DATA}/{n}.csv")]
if sets:
    run(f"python bias_check.py --data {' '.join(sets)} --out {OUT}/bias_check.md")

## 7. Image detector: CLIP ViT-L/14

In [ ]:
if TRAIN_IMAGE and not done("image_detector"):
    per_class = 300 if QUICK_TEST else IMG_MAX_PER_CLASS
    run(f"python train_image_detector.py --data {DATA}/cifake --out {OUT}/image_detector "
        f"--clip {CLIP_MODEL} --real-dir real --fake-dir fake --max-per-class {per_class} "
        f"--epochs {10 if QUICK_TEST else 30}")

## 8. English model: DeBERTa-v3-large (proposed)
If the loss shows `nan`, add `--no-fp16` to the command below.

In [ ]:
if TRAIN_ENGLISH and not done("text_en"):
    limits = "--max-train-rows 2000 --max-test-rows 500 --epochs 1" if QUICK_TEST else \
             f"--max-train-rows {EN_MAX_TRAIN_ROWS} --epochs 2"
    run(f"python train_text_classifier.py --model {EN_MODEL} --train {DATA}/en_train.csv "
        f"--test {DATA}/en_test.csv --out {OUT}/text_en --checkpoint-dir {CKPT}/text_en "
        f"--max-length 256 --batch-size 8 --grad-accum 4 --lr 1e-5 {limits}")

## 9. English baseline: BERT-base (for the comparison table)

In [ ]:
if TRAIN_BASELINE and not done("bert_baseline"):
    limits = "--max-train-rows 2000 --max-test-rows 500 --epochs 1" if QUICK_TEST else \
             f"--max-train-rows {EN_MAX_TRAIN_ROWS} --epochs 3"
    run(f"python train_text_classifier.py --model {BASELINE_MODEL} --train {DATA}/en_train.csv "
        f"--test {DATA}/en_test.csv --out {OUT}/bert_baseline --checkpoint-dir {CKPT}/bert_baseline "
        f"--max-length 256 --batch-size 16 --grad-accum 2 --lr 2e-5 {limits}")

## 10. Telugu model: IndicBERT v2

In [ ]:
if TRAIN_TELUGU and not done("text_te"):
    limits = "--epochs 3" if QUICK_TEST else "--epochs 4"
    run(f"python train_text_classifier.py --model {TE_MODEL} --train {DATA}/te_train.csv "
        f"--test {DATA}/te_test.csv --out {OUT}/text_te --checkpoint-dir {CKPT}/text_te "
        f"--max-length 256 --batch-size 16 --grad-accum 2 --lr 3e-5 {limits}")

## 11. Results table (copy into the report and paper)

In [ ]:
names = {"bert_baseline": "BERT-base (baseline)", "text_en": "DeBERTa-v3-large (proposed)",
         "text_te": "IndicBERT v2 (Telugu)", "image_detector": "CLIP ViT-L/14 (images)"}
rows = ["| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |", "|---|---|---|---|---|---|"]
for key, title in names.items():
    f = f"{OUT}/{key}/metrics.json"
    if os.path.exists(f):
        m = json.load(open(f))
        rows.append(f"| {title} | {m['accuracy']:.4f} | {m['precision']:.4f} | {m['recall']:.4f} "
                    f"| {m['f1']:.4f} | {m['roc_auc']:.4f} |")
table = "\n".join(rows)
print(table)
open(f"{OUT}/results_table.md", "w").write(table + "\n")
if os.path.exists(f"{OUT}/bias_check.md"):
    print("\n" + open(f"{OUT}/bias_check.md").read())
if QUICK_TEST:
    print("\nℹ️ QUICK TEST numbers are NOT for the report. Set QUICK_TEST = False for real results.")

from IPython.display import Image, display
for png in sorted(glob.glob(f"{OUT}/*/confusion_matrix.png")):
    print(png)
    display(Image(png, width=380))

## 12. Package the models for the app
Creates `truthlens_models.zip`. On your PC, **unzip it into `truthlens-ai/backend/models/`**.
The app's Home page then shows the models as **Ready**.

In [ ]:
app_models = [m for m in ("text_en", "text_te", "image_detector") if os.path.exists(f"{OUT}/{m}")]
zip_path = f"{BASE}/truthlens_models{'_quick' if QUICK_TEST else ''}"
staging = f"{WORK}/zip_staging"
shutil.rmtree(staging, ignore_errors=True)
for m in app_models:
    shutil.copytree(f"{OUT}/{m}", f"{staging}/{m}", ignore=shutil.ignore_patterns("training_args.bin"))
for extra in ("results_table.md", "bias_check.md"):
    if os.path.exists(f"{OUT}/{extra}"):
        shutil.copy(f"{OUT}/{extra}", staging)
shutil.make_archive(zip_path, "zip", staging)
print("✅ Created", zip_path + ".zip", "with:", ", ".join(app_models))
print("   Also included: metrics.json, confusion_matrix.png, classification_report.txt, results_table.md, bias_check.md")
if ON_KAGGLE:
    print("Download it from the notebook's Output tab: truthlens/truthlens_models.zip")
else:
    try:
        from google.colab import files
        files.download(zip_path + ".zip")
    except Exception as e:
        print("Download it from the Files panel (or Google Drive → truthlens). Auto-download unavailable:", e)